# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [8]:
links = fetch_website_links("https://nike.com")
links

['#skip-to-content',
 'https://www.nike.com/accessibility#introduction',
 'https://www.nike.com/jordan',
 'https://www.nike.com/w/converse-akmjx',
 'https://www.nike.com/retail',
 'https://www.nike.com/help',
 'https://www.nike.com/help',
 'https://www.nike.com/orders/details/',
 'https://www.nike.com/help/a/shipping-delivery',
 'https://www.nike.com/help/a/returns-policy',
 'https://www.nike.com/help/a/change-cancel-order',
 'https://www.nike.com/help/a/size-charts',
 'https://www.nike.com/help/#contact',
 'https://www.nike.com/membership',
 'https://www.nike.com/promo-code',
 'https://www.nike.com/product-advice',
 '#site-feedback',
 'https://www.nike.com/membership',
 'https://www.nike.com/register',
 'https://www.nike.com',
 'https://www.nike.com/men',
 'https://www.nike.com/w/new-mens-3n82yznik1',
 'https://www.nike.com/w/new-mens-3n82yznik1',
 'https://www.nike.com/w/mens-best-76m50znik1',
 'https://www.nike.com/w/new-upcoming-drops-k0gk',
 'https://www.nike.com/w/medal-stand-col

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.

Example Input: ["/", "/about", "/privacy", "/careers", "mailto:hi@co.com"]
Example Output:
{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "careers page", "url": "https://company.com/careers"}
    ]
}

You should respond in JSON as in the example above.
"""


In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://nike.com"))


Here is the list of links on the website https://nike.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#skip-to-content
https://www.nike.com/accessibility#introduction
https://www.nike.com/jordan
https://www.nike.com/w/converse-akmjx
https://www.nike.com/retail
https://www.nike.com/help
https://www.nike.com/help
https://www.nike.com/orders/details/
https://www.nike.com/help/a/shipping-delivery
https://www.nike.com/help/a/returns-policy
https://www.nike.com/help/a/change-cancel-order
https://www.nike.com/help/a/size-charts
https://www.nike.com/help/#contact
https://www.nike.com/membership
https://www.nike.com/promo-code
https://www.nike.com/product-advice
#site-feedback
https://www.nike.com/membership
https://www.nike.com/register
https://www.nike.com
https://www.nike.com/men
https://www.nike.com/w/new

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [14]:
select_relevant_links("https://nike.com")

{'links': [{'type': 'about page', 'url': 'https://about.nike.com/en'},
  {'type': 'stories page', 'url': 'https://www.nike.com/stories'},
  {'type': 'sustainability page',
   'url': 'https://www.nike.com/sustainability'},
  {'type': 'purpose page', 'url': 'https://purpose.nike.com/'},
  {'type': 'investor relations', 'url': 'https://investors.nike.com/'},
  {'type': 'press / news', 'url': 'https://news.nike.com/'},
  {'type': 'careers page', 'url': 'https://jobs.nike.com/'},
  {'type': 'corporate sales', 'url': 'https://www.nike.com/corporate-sales'},
  {'type': 'impact statement (forced labor)',
   'url': 'https://about.nike.com/en/impact-resources/statement-on-forced-labor'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [16]:
select_relevant_links("https://nike.com")

Selecting relevant links for https://nike.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://about.nike.com/en'},
  {'type': 'about page',
   'url': 'https://about.nike.com/en/impact-resources/statement-on-forced-labor'},
  {'type': 'careers page', 'url': 'https://jobs.nike.com/'},
  {'type': 'stories page', 'url': 'https://www.nike.com/stories'},
  {'type': 'sustainability page',
   'url': 'https://www.nike.com/sustainability'},
  {'type': 'accessibility page',
   'url': 'https://www.nike.com/us/en/accessibility/#introduction'},
  {'type': 'purpose page', 'url': 'https://purpose.nike.com/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://nike.com"))

Selecting relevant links for https://nike.com by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Nike. Just Do It. Nike.com

Skip to main content
Accessibility at Nike
Find a Store
Help
Help
Order Status
Shipping & Delivery
Returns
Order Cancellation
Size Charts
Contact Us
Membership
Promotions & Discounts
Product Advice
Send Us Feedback
Join Us
Sign In
Men
New & Featured
New Arrivals
Best Sellers
Latest Drops
Team USA Collection
SNKRS Launch Calendar
Shop All Sale
Shoes
All Shoes
Basketball
Football
Jordan
Lifestyle
Retro Running
Running
Shoes $100 & Under
Soccer
Training & Gym
Custom Shoes
Clothing
All Clothing
Hoodies & Sweatshirts
Jordan
Matching Sets
Outerwear
Pants
Shorts
Sweatpants
Tops & Graphic Tees
Accessories
Bags & Backpacks
Belts
Hats & Headwear
Socks
Sunglasses
Underwear
Recovery Collection
Travel Collection
Women
New & Featured
New Arrivals
Best Sellers
Latest Drops
Team USA Collection
Nike Style Guide
SNKRS Launch Calendar
Shop All Sale
Shop by Color
Dark Ne

In [30]:
BROCHURE_STYLES = {
    "Professional": "Polished corporate communications expert focusing on value proposition and reliability.",
    # "Creative": "Witty copywriter using engaging analogies and a conversational tone.",
    # "Snarky": "Cynical but informative assistant who cuts through corporate jargon with sharp wit.",
    # "Technical": "Specialist focusing on architectural details and engineering culture."
}


In [31]:
def get_brochure_system_prompt(style="Professional", language="English"):
    # Fix: Get the description string from the dictionary
    style_desc = BROCHURE_STYLES.get(style, BROCHURE_STYLES["Professional"])
    return f"""
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.

Your Style: {style_desc}
Your Language: {language}. Respond ENTIRELY in {language}, including all headers.

Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [32]:
get_brochure_system_prompt("Nike Sport", "https://nike.com")

'\nYou are an assistant that analyzes the contents of several relevant pages from a company website\nand creates a short brochure about the company for prospective customers, investors and recruits.\n\nYour Style: Polished corporate communications expert focusing on value proposition and reliability.\nYour Language: https://nike.com. Respond ENTIRELY in https://nike.com, including all headers.\n\nRespond in markdown without code blocks.\nInclude details of company culture, customers and careers/jobs if you have the information.\n'

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [33]:
def stream_brochure(company_name, url, style="Professional", language="English"):
    # 1. Generate the system prompt (the instructions)
    system_msg = get_brochure_system_prompt(style, language)
    
    # 2. Generate the user prompt (the data / company content)
    # We use our scraper to get all the data
    company_data = fetch_page_and_all_relevant_links(url)
    user_msg = f"Company: {company_name}\nURL: {url}\n\n{company_data}"
    
    # 3. Call the API (Fix: use 'gpt-4o-mini' and pass strings for content)
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
          ],
        stream=True
    )    
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        response += content
        update_display(Markdown(response), display_id=display_handle.display_id)


In [34]:
stream_brochure("Nike Sport", "https://nike.com")

Selecting relevant links for https://nike.com by calling gpt-5-nano
Found 7 relevant links


# Nike Sport Brochure

### Company Overview
Nike, Inc. is a global leader in the sportswear industry, dedicated to bringing inspiration and innovation to every athlete in the world. Our mission encapsulates the belief that everyone is an athlete, a core principle that drives our innovation, product development, and community engagement. With over 50 years of experience, we serve a diverse range of customers, from recreational athletes to professional sports teams, empowering them to achieve their goals and realize their dreams.

### Value Proposition
Nike stands at the forefront of sport and activity, recognized for our commitment to quality, performance, and sustainability. We embrace cutting-edge technology and design to deliver products that enhance athletic performance and meet the evolving needs of our consumers. Our products span various categories including:

- **Footwear**: Athletic and lifestyle shoes across sports such as basketball, running, and soccer.
- **Apparel**: High-performance clothing for men, women, and kids, designed for comfort and functionality.
- **Equipment**: Essential accessories and gear to support training and performance.

### Commitment to Sustainability
Nike is not only focused on innovation but also on creating a better world. We are committed to sustainability and have made significant strides in reducing our environmental impact. Key initiatives include:

- **Renewable Energy**: Achieving over 96% electricity consumption from renewable sources.
- **Waste Reduction**: Diverting 100% of operational waste from landfills and emphasizing recycling.

### Company Culture
At Nike, our culture is built around a shared passion for sport and a commitment to excellence. We cultivate a collaborative and inclusive environment where every employee is encouraged to bring their ideas to the table. Our values include:

- **Serve Athletes**: Empowering individuals through sports and creating opportunities for all.
- **Innovate Fearlessly**: Embracing risk and creativity to develop groundbreaking products.
- **Act with Integrity**: Upholding high standards of ethics and accountability in everything we do.

### Community Impact
We aim to improve access to sports for underserved communities through various initiatives. Our programs are designed to provide quality coaching, promote youth participation, and create safe environments for athletes of all ages.

### Career Opportunities
Joining Nike means becoming part of a dynamic and diverse workforce where your ideas can make a difference. Whether you're interested in product design, marketing, supply chain management, or retail, we offer a variety of career paths. Our **Nike Internship Experience** is a gateway for young professionals, providing real-world experience and mentorship to foster career growth.

### Conclusion
Nike is more than just a brand; it’s a community of athletes and a culture of innovation. Join us in our mission to inspire and elevate the sport experience for everyone. Whether as a customer, investor, or potential recruit, Nike welcomes you to be part of our journey in redefining the future of sport.

For more information, please visit [Nike.com](https://nike.com).